# Relation layers as arc diagrams

`RelationVisualiser` draws the relations of an EstNLTK `RelationLayer` as arcs between
span boxes. The examples follow EstNLTK's `relation_layer.ipynb` tutorial.

By default each relation is a star from its first filled span (in the layer's
`span_names` order) to its other spans, labelled with the target span's name.
Sentences linked by relations are drawn in one diagram.

In [1]:
from estnltk import Text
from estnltk_core import RelationLayer

from estnltk_syntax_vis import RelationVisualiser

## Coreference

Two spans per relation (`mention` → `entity`). The same entity span is one box with
several incoming arcs; "see" is in the second sentence, so both sentences are drawn
together.

In [2]:
text = Text(
    'Mari kirjeldas õhinal, kuidas ta väiksena "Sipsikut" luges: "Ma ei suutnud '
    'seda raamatut kohe kuidagi käest ära panna! Nii põnev oli see!"'
).tag_layer(["words", "sentences"])
coref = RelationLayer("coreference", span_names=["mention", "entity"], text_object=text)
coref.add_annotation(mention=(30, 32), entity=(0, 4))
coref.add_annotation(mention=(61, 63), entity=(0, 4))
coref.add_annotation(mention=(75, 88), entity=(42, 52))
coref.add_annotation(mention=(133, 136), entity=(42, 52))
text.add_layer(coref)

RelationVisualiser()(text.coreference)

entity 
 entity 
 entity 
 entity 
 Mari 
 kirjeldas 
 õhinal 
 , 
 kuidas 
 ta 
 väiksena 
 " 
 Sipsikut 
 " 
 luges 
 : 
 " 
 Ma 
 ei 
 suutnud 
 seda 
 raamatut 
 kohe 
 kuidagi 
 käest 
 ära 
 panna 
 ! 
 Nii 
 põnev 
 oli 
 see 
 ! 
 "

One relation, with one word of context around its spans; skipped words become "…":

In [3]:
RelationVisualiser()(text.coreference, relation=text.coreference[3], window=1)

entity 
 … 
 väiksena 
 " 
 Sipsikut 
 " 
 luges 
 … 
 oli 
 see 
 ! 
 …

## Temporal relations

Arc labels from a relation attribute (`label="rel_type"`).

In [4]:
text = Text(
    "Pühapäeva varahommikul kutsuti politsei Riia mäele. Seal oli ühelt noorelt "
    "mehelt ära võetud nahktagi ja käekell. Juhtumi kohta algatati uurimine."
).tag_layer(["words", "sentences"])
tlinks = RelationLayer(
    "temporal_relations",
    span_names=["entity_a", "entity_b"],
    attributes=["rel_type"],
    text_object=text,
)
tlinks.add_annotation(entity_a=(0, 22), entity_b=(23, 30), rel_type="INCLUDES")
tlinks.add_annotation(entity_a=(82, 92), entity_b=(23, 30), rel_type="BEFORE")
tlinks.add_annotation(entity_a=(82, 92), entity_b=(114, 121), rel_type="IDENTITY")
tlinks.add_annotation(entity_a=(82, 92), entity_b=(128, 145), rel_type="BEFORE")
text.add_layer(tlinks)

RelationVisualiser(label="rel_type")(text.temporal_relations)

INCLUDES 
 BEFORE 
 IDENTITY 
 BEFORE 
 Pühapäeva 
 varahommikul 
 kutsuti 
 politsei 
 Riia 
 mäele 
 . 
 Seal 
 oli 
 ühelt 
 noorelt 
 mehelt 
 ära 
 võetud 
 nahktagi 
 ja 
 käekell 
 . 
 Juhtumi 
 kohta 
 algatati 
 uurimine 
 .

## Semantic roles

The tutorial's layer has no span for the verb (the predicate is in the attribute
`rel`), so the star starts from `arg0`. `hub_label="rel"` shows the attribute under
the hub span, and `rows` adds rows from other layers under the words.

In [5]:
text = Text(
    "President Bush kohtus temaga privaatselt Valges Majas teisipäeval. Aga John ja "
    "Mari kohtusid hoopis kokteilipeol. Mari ei ostnud Johnile kokteili."
).tag_layer(["words", "sentences", "morph_analysis"])
roles = RelationLayer(
    "semantic_roles",
    span_names=[
        "arg0",
        "arg1",
        "arg2",
        "arg3",
        "arg4",
        "argm_mnr",
        "argm_tmp",
        "argm_loc",
    ],
    attributes=["rel"],
    text_object=text,
)
roles.add_annotation(
    arg0=(0, 14),
    arg1=(22, 28),
    argm_mnr=(29, 40),
    argm_tmp=(54, 65),
    argm_loc=(41, 53),
    rel="kohtumine",
)
roles.add_annotation(arg0=(71, 75), arg1=(79, 83), rel="kohtumine")
roles.add_annotation(
    arg0=(114, 118), arg1=(137, 145), arg4=(129, 136), rel="ostmine-NEG"
)
text.add_layer(roles)

vis = RelationVisualiser(hub_label="rel", rows=[("morph_analysis", "partofspeech")])
vis(text.semantic_roles)

kohtumine 
 
 
 
 
 
 
 
 
 arg1 
 argm_mnr 
 argm_tmp 
 argm_loc 
 President 
 S 
 Bush 
 H 
 kohtus 
 V 
 temaga 
 P 
 privaatselt 
 D 
 Valges 
 S 
 Majas 
 H 
 teisipäeval 
 S 
 . 
 Z 
 
 partofspeech 
 
 
 
 
 
 kohtumine 
 
 
 arg1 
 Aga 
 J 
 John 
 H 
 ja 
 J 
 Mari 
 H 
 kohtusid 
 V 
 hoopis 
 D 
 kokteilipeol 
 S 
 . 
 Z 
 
 partofspeech 
 
 
 
 
 
 ostmine-NEG 
 
 
 
 
 arg1 
 arg4 
 Mari 
 H 
 ei 
 V 
 ostnud 
 V 
 Johnile 
 H 
 kokteili 
 S 
 . 
 Z 
 
 partofspeech

With a `verb` span (as in the PropBank pre-annotator's layers) the default star
starts from the verb. `hub="verb"` makes this explicit. `highlight` emphasises a
relation and `window=0` keeps only its spans; `span_names=True` writes the span
names into the boxes.

In [6]:
text = Text("President Bush kohtus temaga privaatselt Valges Majas teisipäeval.")
text.tag_layer(["words", "sentences", "morph_analysis"])
frames = RelationLayer(
    "frames",
    span_names=["verb", "arg0", "arg1", "argm_mnr", "argm_tmp", "argm_loc"],
    attributes=["sense_id"],
    text_object=text,
)
frames.add_annotation(
    verb=(15, 21),
    arg0=(0, 14),
    arg1=(22, 28),
    argm_mnr=(29, 40),
    argm_tmp=(54, 65),
    argm_loc=(41, 53),
    sense_id="kohtuma_1",
)
text.add_layer(frames)

vis = RelationVisualiser(hub="verb", hub_label="sense_id", span_names=True)
vis(text.frames, highlight=text.frames[0], window=0)

arg0 
 
 verb, kohtuma_1 
 
 arg1 
 
 argm_mnr 
 
 argm_loc 
 
 argm_tmp 
 
 
 
 
 
 arg0 
 arg1 
 argm_mnr 
 argm_tmp 
 argm_loc 
 President 
 Bush 
 kohtus 
 temaga 
 privaatselt 
 Valges 
 Majas 
 teisipäeval 
 …

## Attaching arcs to syntactic heads

With a syntax layer, `attach="head"` attaches arcs to the head word of each span
instead of its centre (here `maltparser_syntax`, which needs Java).

In [7]:
text.tag_layer("maltparser_syntax")
vis = RelationVisualiser(hub="verb", attach="head", syntax_layer="maltparser_syntax")
vis(text.frames)

arg0 
 arg1 
 argm_mnr 
 argm_tmp 
 argm_loc 
 President 
 Bush 
 kohtus 
 temaga 
 privaatselt 
 Valges 
 Majas 
 teisipäeval 
 .

## Saving an SVG

In [8]:
svg = RelationVisualiser(hub="verb").svg(text.frames)
print(svg[:120], "...")

<svg xmlns="http://www.w3.org/2000/svg" width="556.9" height="138.5" viewBox="0 0 556.9 138.5" font-family="Arial, Helve ...
